# Financial Tweets Sentiment Analysis

This notebook presents a complete analysis workflow for the current project:

- Data loading and schema checks
- Label distribution and text-feature EDA
- `TF-IDF + SGDClassifier(log_loss)` baseline
- An optional Transformer training entry point
- Evaluation, slice analysis, and error case review

Note: the production training, prediction, and evaluation logic now lives in the `financial_tweets_sentiment_analysis/` package. This notebook is mainly for exploration and presentation.


## 1. Setup

Start by importing the project code and setting notebook display options.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from financial_tweets_sentiment_analysis import data, features, predict, train

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
PROJECT_ROOT


## 2. Load Data

Use the current project data interface here instead of the old `title/description/tag` tutorial schema.


In [ ]:
dataset_path = PROJECT_ROOT / 'datasets' / 'processed' / 'train' / 'train_sample_5000.csv'
df = data.load_data(str(dataset_path))
print(df.shape)
df.head()


## 3. Schema Check

Confirm that the columns match the current project contract.


In [ ]:
df.dtypes


In [ ]:
df[['id', 'tweet_raw', 'tweet_clean', 'sentiment', 'source', 'ticker_mentions', 'has_url']].head(10)


## 4. EDA: Label Distribution

First inspect whether the three sentiment labels are reasonably balanced.


In [ ]:
label_counts = df['sentiment'].value_counts().rename_axis('sentiment').reset_index(name='count')
label_counts


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=label_counts, x='sentiment', y='count', palette='Set2')
plt.title('Financial Tweet Sentiment Distribution')
plt.show()


## 5. EDA: Text and Market-specific Signals

Beyond label distribution, we also care about text length, cashtags, URLs, and data source patterns.


In [ ]:
eda_df = df.copy()
eda_df['tweet_length'] = eda_df['tweet_clean'].str.split().str.len()
eda_df['num_tickers'] = eda_df['ticker_mentions'].apply(len)
eda_df[['tweet_length', 'num_tickers', 'has_url']].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(eda_df['tweet_length'], bins=30, ax=axes[0], color='#2E86AB')
axes[0].set_title('Tweet Length Distribution')
sns.countplot(data=eda_df, x='has_url', ax=axes[1], palette='pastel')
axes[1].set_title('URL Presence')
sns.histplot(eda_df['num_tickers'], bins=15, ax=axes[2], color='#F18F01')
axes[2].set_title('Ticker Mentions per Tweet')
plt.tight_layout()
plt.show()


In [ ]:
source_counts = df['source'].value_counts().head(10)
source_counts


## 6. Train / Validation / Test Split

The production code uses the same split logic.


In [ ]:
train_df, val_df, test_df = data.split_dataframe(df)
train_df.shape, val_df.shape, test_df.shape


In [ ]:
pd.concat(
    {
        'train': train_df['sentiment'].value_counts(normalize=True),
        'val': val_df['sentiment'].value_counts(normalize=True),
        'test': test_df['sentiment'].value_counts(normalize=True),
    },
    axis=1,
).fillna(0)


## 7. Baseline Model

To make the project more complete, start with a traditional NLP baseline: `tweet_clean -> TF-IDF -> linear classifier`.


In [ ]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
clf = SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42)

X_train = vectorizer.fit_transform(train_df['tweet_clean'])
X_test = vectorizer.transform(test_df['tweet_clean'])
clf.fit(X_train, train_df['sentiment'])
baseline_pred = clf.predict(X_test)

print(classification_report(test_df['sentiment'], baseline_pred, digits=4))


In [ ]:
cm = confusion_matrix(test_df['sentiment'], baseline_pred, labels=['bearish', 'bullish', 'neutral'])
cm_df = pd.DataFrame(cm, index=['true_bearish', 'true_bullish', 'true_neutral'], columns=['pred_bearish', 'pred_bullish', 'pred_neutral'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Baseline Confusion Matrix')
plt.show()


## 8. Package-level Training API

If you want to train the baseline through the package API, call `train.train_model(...)` directly.

The next cell is disabled by default so the notebook does not retrain every time it is opened.


In [ ]:
RUN_BASELINE_TRAINING = False

if RUN_BASELINE_TRAINING:
    baseline_summary = train.train_model(
        model_type='baseline',
        dataset_loc=str(dataset_path),
        num_samples=1500,
        run_name='notebook-baseline-demo',
    )
    baseline_summary


## 9. Optional Transformer Experiment

The Transformer path corresponds to the second modeling track in the project.

Only enable it after `torch` and `transformers` are installed locally.


In [ ]:
RUN_TRANSFORMER_TRAINING = False

if RUN_TRANSFORMER_TRAINING:
    transformer_summary = train.train_model(
        model_type='transformer',
        dataset_loc=str(dataset_path),
        num_samples=1000,
        num_epochs=1,
        batch_size=8,
        transformer_model_name='ProsusAI/finbert',
        run_name='notebook-finbert-demo',
    )
    transformer_summary


## 10. Inference Demo

After training, you can use the unified prediction interface.


In [ ]:
sample_tweets = [
    '$AAPL looks strong into earnings, momentum is building.',
    'Fed comments keep markets on edge, not seeing a clear trend yet.',
    '$TSLA is getting crushed again, this breakdown looks ugly.'
]

# Replace this with a saved run_id if you already trained a model.
RUN_ID = 'notebook-baseline-demo'


In [ ]:
if RUN_ID and (PROJECT_ROOT / 'artifacts' / 'model_registry' / RUN_ID).exists():
    predict.predict_texts(RUN_ID, sample_tweets)
else:
    print('No saved run found yet. Train a baseline first or change RUN_ID.')


## 11. Slice Analysis

This section applies the same slice-analysis ideas from the project to inspect localized test-set behavior.


In [ ]:
analysis_df = test_df.copy()
analysis_df['is_short_text'] = analysis_df['tweet_clean'].apply(features.is_short_text)
analysis_df['has_ticker'] = analysis_df['ticker_mentions'].apply(bool)
analysis_df['is_news_headline'] = analysis_df['tweet_raw'].apply(features.is_news_headline)
analysis_df[['is_short_text', 'has_ticker', 'has_url', 'is_news_headline']].mean().sort_values(ascending=False)


## 12. Error Analysis

In a portfolio project, error analysis is often more valuable than reporting a single accuracy number.


In [ ]:
error_df = test_df[['tweet_raw', 'tweet_clean', 'sentiment', 'ticker_mentions', 'has_url', 'source']].copy()
error_df['prediction'] = baseline_pred
error_df = error_df[error_df['sentiment'] != error_df['prediction']].copy()
error_df['tweet_length'] = error_df['tweet_clean'].str.split().str.len()
error_df.sort_values('tweet_length', ascending=False).head(15)


## 13. What to Highlight in the Final Project

When presenting the final project, highlight:

- Financial-text characteristics such as cashtags, URLs, headlines, and mixed subjective opinions
- The gap between the baseline and the transformer model
- Which samples are hardest, such as bullish/bearish confusions, news summaries, and sarcastic phrasing
- Current limitations, including heterogeneous data sources, label noise, and market time sensitivity

If this notebook is part of your portfolio, run the key cells and save the outputs so readers can immediately see the figures and example results.
